# 06 — Final model, SHAP and the top-20 retrain

The main result. NHANES and KNHANES are pooled into a single training set of
26,798 participants, three boosted-tree models and a soft-voting ensemble are
trained on 70% of it, and SHAP is used to ask which features the best model
actually relies on. A final CatBoost is then retrained on just the twenty most
influential features, to see how much of the performance survives a twenty-fold
reduction in feature count.

The twenty features are **derived here** from the SHAP ranking rather than
written down as a list. That matters because SHAP's ranking and CatBoost's own
`feature_importances_` disagree from the third entry onwards, so which one the
list came from is not something to leave implicit.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle

import numpy as np
import polars as pl

from src.data.io import load_hyperparameters, output_path, processed_path, repo_root
from src.features.selection import drop_target_derived
from src.logging_utils import configure_logging
from src.models.evaluate import compute_metrics
from src.models.explain import (
    COLOR_BLIND_CMAP,
    plot_shap_summary,
    rank_features,
    shap_values,
)
from src.models.train import (
    build_classifier,
    build_voting_classifier,
    split_features_target,
    split_frames,
    stratified_split,
)
from src.viz.figures import add_panel_label, plot_confusion_matrix, plot_roc_pr_curves

configure_logging(ROOT / "logs")

ALGORITHMS = ["CatBoost", "XGBoost", "lightGBM"]
PARAMS = load_hyperparameters()["pooled"]

# KNHANES first. The concatenation order sets the row order, which sets which
# participants land in the training and the test split.
pooled = drop_target_derived(
    pl.concat(
        [
            pl.read_parquet(processed_path("KNHANES_race2_features.parquet")),
            pl.read_parquet(processed_path("NHANES_race1_features.parquet")),
        ]
    )
)
X, y = split_features_target(pooled)
X_train, X_test, y_train, y_test = stratified_split(X, y)
print(f"pooled {pooled.shape}, IR+ {pooled['IR'].mean():.4f}")
print(f"train {X_train.shape}, test {X_test.shape}")

pooled (26798, 242), IR+ 0.3511
train (18758, 241), test (8040, 241)


## The pooled model

Three libraries with their tuned hyperparameters, then a soft-voting ensemble
over the three. Class imbalance is handled by weighting, and the decision
threshold is fixed at 0.5.

In [2]:
runs = {}
rows = []

fitted = []
for algorithm in ALGORITHMS:
    model = build_classifier(algorithm, y_train, PARAMS[algorithm])
    model.fit(X_train, y_train)
    fitted.append((algorithm, model))

ensemble = build_voting_classifier(
    [(f"model_{index}", model) for index, (_, model) in enumerate(fitted)]
)
ensemble.fit(X_train, y_train)
fitted.append(("Voting", ensemble))

for algorithm, model in fitted:
    preds = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, preds)
    runs[algorithm] = {
        "model": model, "preds": preds, "metrics": metrics,
        "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
    }
    rows.append(
        {
            "feature_set": "all features (241)",
            "model": algorithm,
            **{k: v for k, v in metrics.items() if k not in ("confusion_matrix", "optimal_threshold")},
            **metrics["confusion_matrix"],
        }
    )

pl.DataFrame(rows)

feature_set,model,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,TP,TN,FP,FN
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64
"""all features (241)""","""CatBoost""",0.88,0.799,0.778,0.81,0.689,0.871,0.731,0.588,0.816,2196,4224,993,627
"""all features (241)""","""XGBoost""",0.879,0.797,0.777,0.808,0.686,0.87,0.729,0.585,0.815,2194,4213,1004,629
"""all features (241)""","""lightGBM""",0.879,0.799,0.779,0.809,0.689,0.871,0.731,0.588,0.815,2199,4223,994,624
"""all features (241)""","""Voting""",0.88,0.799,0.782,0.809,0.689,0.873,0.732,0.591,0.817,2207,4220,997,616


## What the model relies on

SHAP attributes every individual prediction to the features that produced it, so
the ranking below is over contributions to predictions rather than over the
model's internal split statistics.

The explainer is given the feature matrix alone — not the label, and not the
model's own predictions. Columns no tree splits on receive exactly zero
attribution, so passing extras would be harmless here, but it would also be
meaningless.

Two versions of the figure are written: the default colour map, and a
colour-blind-friendly one using Okabe–Ito blue for low feature values and orange
for high. Red and blue, the library default, are the pair hardest to separate
under the common forms of colour vision deficiency.

In [3]:
values = shap_values(runs["CatBoost"]["model"], X_test)
ranked = rank_features(values, list(X_test.columns))
top_20 = ranked[:20]

plot_shap_summary(values, X_test, output_path("SHAP_summary_plot_new.png"))
plot_shap_summary(
    values,
    X_test,
    output_path("SHAP_summary_plot_new_color_blind.png"),
    cmap=COLOR_BLIND_CMAP,
)

for position, name in enumerate(top_20, start=1):
    print(f"{position:2d}. {name}")


2026-09-22 19:48:06 [INFO] src.models.explain: Explaining with fasttreeshap


2026-09-22 19:48:06 [INFO] src.models.explain: SHAP values: (8040, 241)


2026-09-22 19:48:07 [INFO] src.models.explain: Wrote output/SHAP_summary_plot_new.png


2026-09-22 19:48:08 [INFO] src.models.explain: Wrote output/SHAP_summary_plot_new_color_blind.png


 1. BMI_mul_FASTING_GLUCOSE
 2. AGE
 3. SEX
 4. BODY_WAISTLINE_mul_FASTING_GLUCOSE
 5. SGOT_div_SGPT
 6. FASTING_GLUCOSE_mul_HBA1C
 7. FASTING_GLUCOSE
 8. BMI_mul_HBA1C
 9. BODY_WAISTLINE_mul_HBA1C
10. FASTING_GLUCOSE_log
11. FASTING_GLUCOSE_div_HBA1C
12. TG_div_T_CHO
13. BMI_mul_TG
14. FASTING_GLUCOSE_div_HDL_C
15. FASTING_GLUCOSE_sqrt
16. FASTING_GLUCOSE_mul_MAP
17. HDL_C_div_SGPT
18. BMI_div_HDL_C
19. SGPT_sqrt
20. BMI_mul_SGPT


## Retraining on the top twenty

Same CatBoost hyperparameters, same split, twenty features instead of 241.

In [4]:
top_20_frame = pooled.select(top_20 + ["IR"])
X20, y20 = split_features_target(top_20_frame)
X20_train, X20_test, y20_train, y20_test = stratified_split(X20, y20)

model_20 = build_classifier("CatBoost", y20_train, PARAMS["CatBoost"])
model_20.fit(X20_train, y20_train)
preds_20 = model_20.predict_proba(X20_test)[:, 1]
metrics_20 = compute_metrics(y20_test, preds_20)

runs["CatBoost_top20"] = {
    "model": model_20, "preds": preds_20, "metrics": metrics_20,
    "X_train": X20_train, "X_test": X20_test, "y_train": y20_train, "y_test": y20_test,
}
rows.append(
    {
        "feature_set": "top 20 by SHAP",
        "model": "CatBoost",
        **{k: v for k, v in metrics_20.items() if k not in ("confusion_matrix", "optimal_threshold")},
        **metrics_20["confusion_matrix"],
    }
)

performance = pl.DataFrame(rows)
performance.to_pandas().to_excel(output_path("final_model_performance.xlsx"), index=False)
performance

feature_set,model,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,TP,TN,FP,FN
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64
"""all features (241)""","""CatBoost""",0.88,0.799,0.778,0.81,0.689,0.871,0.731,0.588,0.816,2196,4224,993,627
"""all features (241)""","""XGBoost""",0.879,0.797,0.777,0.808,0.686,0.87,0.729,0.585,0.815,2194,4213,1004,629
"""all features (241)""","""lightGBM""",0.879,0.799,0.779,0.809,0.689,0.871,0.731,0.588,0.815,2199,4223,994,624
"""all features (241)""","""Voting""",0.88,0.799,0.782,0.809,0.689,0.873,0.732,0.591,0.817,2207,4220,997,616
"""top 20 by SHAP""","""CatBoost""",0.88,0.799,0.785,0.807,0.687,0.874,0.733,0.591,0.815,2215,4208,1009,608


Dropping from 241 features to 20 costs nothing measurable — AUC is unchanged at
0.880 — and sensitivity improves slightly. That is the practical result: a model
needing twenty routinely derived quantities performs as well as one needing all
241.

In [5]:
confusion = plot_confusion_matrix(
    runs["CatBoost"]["metrics"]["confusion_matrix"],
    "CatBoost",
    output_path("ConfusionMatrix_CatBoost.png"),
)
roc_pr = plot_roc_pr_curves(
    y_test, runs["CatBoost"]["preds"], "CatBoost", output_path("ROC_PR_CatBoost.png")
)

# The two curves and the matrix are panels (a) and (b) of one manuscript figure,
# so each is also written with its panel letter in a strip above the plot.
add_panel_label(roc_pr, "(a)", output_path("ROC_PR_CatBoost_a.png"))
add_panel_label(confusion, "(b)", output_path("ConfusionMatrix_CatBoost_b.png"))
print("figures written")

2026-09-22 19:48:09 [INFO] src.viz.figures: Wrote output/ConfusionMatrix_CatBoost.png


2026-09-22 19:48:10 [INFO] src.viz.figures: Wrote output/ROC_PR_CatBoost.png


2026-09-22 19:48:10 [INFO] src.viz.figures: Wrote output/ROC_PR_CatBoost_a.png


2026-09-22 19:48:10 [INFO] src.viz.figures: Wrote output/ConfusionMatrix_CatBoost_b.png


figures written


## Saving the models

Both CatBoost models are kept. The top-20 model is the deployable one and the
model that scores Taiwan Biobank; the all-features model is the one the SHAP
figure and the 241-feature performance row describe, so it is worth keeping so
that neither result has to be regenerated to be inspected.

In [6]:
import json

models_dir = repo_root() / "models"
models_dir.mkdir(exist_ok=True)

with open(models_dir / "catboost_all_features.pkl", "wb") as handle:
    pickle.dump(runs["CatBoost"]["model"], handle)
with open(models_dir / "catboost_top20.pkl", "wb") as handle:
    pickle.dump(runs["CatBoost_top20"]["model"], handle)

manifest = pl.DataFrame(
    [
        {
            "file": "catboost_all_features.pkl",
            "training_data": "NHANES + KNHANES pooled",
            "n_features": X_train.shape[1],
            "n_train": len(X_train),
            "n_test": len(X_test),
            "hyperparameters": json.dumps(PARAMS["CatBoost"]),
            **{k: runs["CatBoost"]["metrics"][k] for k in ("roc_auc", "sensitivity(recall)", "NPV")},
        },
        {
            "file": "catboost_top20.pkl",
            "training_data": "NHANES + KNHANES pooled",
            "n_features": X20_train.shape[1],
            "n_train": len(X20_train),
            "n_test": len(X20_test),
            "hyperparameters": json.dumps(PARAMS["CatBoost"]),
            **{k: metrics_20[k] for k in ("roc_auc", "sensitivity(recall)", "NPV")},
        },
    ]
)
manifest.to_pandas().to_csv(models_dir / "manifest.csv", index=False)
manifest

file,training_data,n_features,n_train,n_test,hyperparameters,roc_auc,sensitivity(recall),NPV
str,str,i64,i64,i64,str,f64,f64,f64
"""catboost_all_features.pkl""","""NHANES + KNHANES pooled""",241,18758,8040,"""{""colsample_bylevel"": 0.988952…",0.88,0.778,0.871
"""catboost_top20.pkl""","""NHANES + KNHANES pooled""",20,18758,8040,"""{""colsample_bylevel"": 0.988952…",0.88,0.785,0.874
